# Quantum-AI Drug Discovery — Inference (Colab)

Run the **MolGAN → VQE** pipeline on Colab CPU/GPU instead of locally.

Workflow:
1. Upload your trained `molgan.pt` (the checkpoint produced by `train_colab.ipynb`).
2. Either sample fresh candidates from the GAN, or paste your own SMILES.
3. Each candidate is scored by VQE (PennyLane Lightning, STO-3G, 2e/2o active space).
4. Top-K lowest-energy molecules are displayed with structures.
5. Results are saved to a CSV you can download.

Why use this notebook:
- Colab CPUs are often faster than laptop CPUs for VQE.
- You can crank up the candidate count without freezing your local machine.
- All the heavy quantum-chem deps (PySCF, basis-set-exchange) are already installed in `pip` form.

## 0. Environment check

VQE on Lightning is CPU-bound; a GPU runtime won't help quantum simulation,
but it does speed up the MolGAN sampling step. Either runtime is fine.

In [ ]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 1. Install dependencies

All of these are reasonably fast to install on Colab (~1–2 minutes total).

In [ ]:
!pip install -q rdkit pennylane pennylane-lightning pyscf basis-set-exchange

## 2. Upload the trained checkpoint

Pick `molgan.pt` (the file the training notebook downloaded for you).

In [ ]:
from pathlib import Path
from google.colab import files

CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('Choose molgan.pt')
uploaded = files.upload()
for name, content in uploaded.items():
    target = CHECKPOINT_DIR / name
    target.write_bytes(content)
    print(f'Saved -> {target} ({len(content):,} bytes)')

CHECKPOINT_PATH = CHECKPOINT_DIR / 'molgan.pt'
assert CHECKPOINT_PATH.exists(), f'Expected {CHECKPOINT_PATH}'

## 3. SMILES ↔ graph tensor encoding

Identical to `src/smiles_graph.py`. Required to load the checkpoint correctly.

In [ ]:
from typing import List, Optional, Tuple

import numpy as np
import torch
from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')

MAX_ATOMS = 12
ATOM_MAP = [6, 7, 8, 9, 16, 17]
PAD_INDEX = len(ATOM_MAP)
ATOM_TYPES = len(ATOM_MAP) + 1
BOND_TYPES = 5

BOND_MAP = [
    None,
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
    Chem.BondType.AROMATIC,
]


def _attempt_recovery(rw):
    try:
        mol = rw.GetMol()
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    except Exception:
        return None
    best = None
    for frag in frags:
        try:
            Chem.SanitizeMol(frag)
        except Exception:
            continue
        if frag.GetNumAtoms() == 0:
            continue
        if best is None or frag.GetNumAtoms() > best.GetNumAtoms():
            best = frag
    return best


def graph_to_mol(nodes, edges):
    node_idx = np.argmax(nodes, axis=-1)
    edge_idx = np.argmax(edges, axis=-1)
    rw = Chem.RWMol()
    rdkit_indices = []
    for v in range(MAX_ATOMS):
        atype = int(node_idx[v])
        if atype == PAD_INDEX:
            rdkit_indices.append(None)
            continue
        rdkit_indices.append(rw.AddAtom(Chem.Atom(ATOM_MAP[atype])))
    for i in range(MAX_ATOMS):
        if rdkit_indices[i] is None:
            continue
        for j in range(i + 1, MAX_ATOMS):
            if rdkit_indices[j] is None:
                continue
            bidx = int(edge_idx[i, j])
            if bidx == 0 or bidx >= len(BOND_MAP) or BOND_MAP[bidx] is None:
                continue
            rw.AddBond(rdkit_indices[i], rdkit_indices[j], BOND_MAP[bidx])
    mol = rw.GetMol()
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return _attempt_recovery(rw)
    if mol.GetNumAtoms() == 0:
        return None
    return mol


def graph_to_smiles(nodes, edges):
    mol = graph_to_mol(nodes, edges)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    try:
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    except Exception:
        return None
    best = None
    for frag in frags:
        try:
            Chem.SanitizeMol(frag)
        except Exception:
            continue
        if frag.GetNumHeavyAtoms() < 2:
            continue
        if best is None or frag.GetNumHeavyAtoms() > best.GetNumHeavyAtoms():
            best = frag
    if best is None:
        return None
    try:
        smi = Chem.MolToSmiles(best, canonical=True)
    except Exception:
        return None
    if not smi or '.' in smi:
        return None
    return smi


def batch_decode_to_smiles(nodes_batch, edges_batch):
    nodes_np = nodes_batch.detach().cpu().numpy()
    edges_np = edges_batch.detach().cpu().numpy()
    return [graph_to_smiles(nodes_np[i], edges_np[i]) for i in range(nodes_np.shape[0])]

print('Encoder ready.')

## 4. MolGAN model + checkpoint loading

In [ ]:
from dataclasses import dataclass
from typing import Optional

import torch.nn as nn


@dataclass
class MolGANConfig:
    z_dim: int = 32
    g_hidden: int = 256
    d_hidden: int = 256
    lr_g: float = 1e-4
    lr_d: float = 1e-4
    betas: tuple = (0.5, 0.9)
    n_critic: int = 5
    gp_lambda: float = 10.0
    batch_size: int = 64


class _Generator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = cfg.g_hidden
        self.trunk = nn.Sequential(
            nn.Linear(cfg.z_dim, h),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(h, h * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(h * 2, h * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.edge_head = nn.Linear(h * 4, MAX_ATOMS * MAX_ATOMS * BOND_TYPES)
        self.node_head = nn.Linear(h * 4, MAX_ATOMS * ATOM_TYPES)

    def forward(self, z):
        h = self.trunk(z)
        e = self.edge_head(h).view(-1, MAX_ATOMS, MAX_ATOMS, BOND_TYPES)
        n = self.node_head(h).view(-1, MAX_ATOMS, ATOM_TYPES)
        e = 0.5 * (e + e.transpose(1, 2))
        return torch.softmax(e, dim=-1), torch.softmax(n, dim=-1)


class _Discriminator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        in_dim = (MAX_ATOMS * MAX_ATOMS * BOND_TYPES) + (MAX_ATOMS * ATOM_TYPES)
        h = cfg.d_hidden
        self.net = nn.Sequential(
            nn.Linear(in_dim, h * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(h * 2, h),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(h, 1),
        )

    def forward(self, edges, nodes):
        x = torch.cat([edges.flatten(1), nodes.flatten(1)], dim=1)
        return self.net(x)


class MolGAN:
    def __init__(self, cfg=None, device=None):
        self.cfg = cfg or MolGANConfig()
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.generator = _Generator(self.cfg).to(self.device)
        self.discriminator = _Discriminator(self.cfg).to(self.device)

    @classmethod
    def load_checkpoint(cls, path, device=None):
        device_obj = torch.device(
            device or ('cuda' if torch.cuda.is_available() else 'cpu')
        )
        payload = torch.load(path, map_location=device_obj, weights_only=False)
        cfg = MolGANConfig(**payload['config'])
        instance = cls(cfg=cfg, device=str(device_obj))
        instance.generator.load_state_dict(payload['generator'])
        instance.discriminator.load_state_dict(payload['discriminator'])
        return instance

    @torch.no_grad()
    def sample_smiles(self, n, max_tries=10):
        self.generator.eval()
        out, seen = [], set()
        batch = max(n * 2, 32)
        for attempt in range(1, max_tries + 1):
            if len(out) >= n:
                break
            z = torch.randn(batch, self.cfg.z_dim, device=self.device)
            edges, nodes = self.generator(z)
            for smi in batch_decode_to_smiles(nodes, edges):
                if not smi or smi in seen:
                    continue
                seen.add(smi)
                out.append(smi)
                if len(out) >= n:
                    break
        return out[:n]

molgan = MolGAN.load_checkpoint(CHECKPOINT_PATH)
print('MolGAN loaded on', molgan.device)

## 5. Quantum chemistry helpers

Mirrors `src/quantum/geometry.py` and `src/quantum/vqe.py`. SMILES → 3D
embedding (RDKit ETKDG + UFF) → PySCF/PennyLane Hamiltonian → hardware-
efficient ansatz (RY + CNOT + RY layers) → Adam optimization.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Optional

import pennylane as qml
from pennylane import numpy as pnp
from rdkit.Chem import AllChem

ANGSTROM_TO_BOHR = 1.8897259886


class GeometryError(RuntimeError):
    pass


def smiles_to_geometry(smiles, optimize=True, seed=42):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise GeometryError(f'Invalid SMILES: {smiles!r}')
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) == -1:
        params.useRandomCoords = True
        if AllChem.EmbedMolecule(mol, params) == -1:
            raise GeometryError(f'ETKDG failed for {smiles!r}')
    if optimize:
        try:
            AllChem.UFFOptimizeMolecule(mol, maxIters=200)
        except Exception:
            pass
    conf = mol.GetConformer()
    symbols, coords = [], []
    for i, atom in enumerate(mol.GetAtoms()):
        symbols.append(atom.GetSymbol())
        pos = conf.GetAtomPosition(i)
        coords.append([pos.x, pos.y, pos.z])
    return symbols, np.asarray(coords, dtype=float) * ANGSTROM_TO_BOHR


class VQEStatus(str, Enum):
    OK = 'ok'
    GEOMETRY_FAILED = 'geometry_failed'
    HAMILTONIAN_FAILED = 'hamiltonian_failed'
    OPTIMIZATION_FAILED = 'optimization_failed'
    MOLECULE_TOO_LARGE = 'molecule_too_large'
    TIMEOUT = 'timeout'


@dataclass
class VQEResult:
    smiles: str
    energy: float
    n_qubits: int
    steps_run: int
    status: VQEStatus
    error: Optional[str] = None

    @property
    def succeeded(self):
        return self.status == VQEStatus.OK


FAILED_ENERGY = float('inf')


# STO-3G basis function counts (used as an SCF-cost proxy).
_STO3G_BFS = {
    'H': 1, 'He': 1,
    'Li': 5, 'Be': 5, 'B': 5, 'C': 5, 'N': 5, 'O': 5, 'F': 5, 'Ne': 5,
    'Na': 9, 'Mg': 9, 'Al': 9, 'Si': 9, 'P': 9, 'S': 9, 'Cl': 9, 'Ar': 9,
}


def _run_with_timeout(fn, timeout, *args, **kwargs):
    """Wall-clock guard around a single function call. Leaks the worker
    thread on timeout (PySCF releases the GIL, so the main loop is freed)."""
    import concurrent.futures
    if not timeout or timeout <= 0:
        return fn(*args, **kwargs)
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(fn, *args, **kwargs)
    try:
        return fut.result(timeout=timeout)
    except concurrent.futures.TimeoutError:
        try:
            ex.shutdown(wait=False, cancel_futures=True)
        except TypeError:
            ex.shutdown(wait=False)
        raise TimeoutError(f'Operation exceeded {timeout:.0f}s')
    else:
        ex.shutdown(wait=True)


class VQEScorer:
    def __init__(self, active_electrons=2, active_orbitals=2, basis='sto-3g',
                 vqe_steps=50, vqe_stepsize=0.1, device_name='lightning.qubit',
                 max_heavy_atoms=6, max_basis_functions=40,
                 hamiltonian_timeout_sec=60.0, optimization_timeout_sec=60.0):
        self.active_electrons = active_electrons
        self.active_orbitals = active_orbitals
        self.basis = basis
        self.vqe_steps = vqe_steps
        self.vqe_stepsize = vqe_stepsize
        self.device_name = device_name
        self.max_heavy_atoms = max_heavy_atoms
        self.max_basis_functions = max_basis_functions
        self.hamiltonian_timeout_sec = hamiltonian_timeout_sec
        self.optimization_timeout_sec = optimization_timeout_sec
        self._cache = {}

    def screen(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False, 'Invalid SMILES'
        n_heavy = mol.GetNumHeavyAtoms()
        if self.max_heavy_atoms and n_heavy > self.max_heavy_atoms:
            return False, f'{n_heavy} heavy atoms exceeds budget of {self.max_heavy_atoms}'
        return True, None

    def score(self, smiles):
        if smiles in self._cache:
            return self._cache[smiles]
        ok, reason = self.screen(smiles)
        if not ok:
            r = VQEResult(smiles, FAILED_ENERGY, 0, 0, VQEStatus.MOLECULE_TOO_LARGE, reason)
            self._cache[smiles] = r
            return r
        try:
            symbols, coords = smiles_to_geometry(smiles)
        except GeometryError as exc:
            r = VQEResult(smiles, FAILED_ENERGY, 0, 0, VQEStatus.GEOMETRY_FAILED, str(exc))
            self._cache[smiles] = r
            return r
        n_bfs = sum(_STO3G_BFS.get(s, 5) for s in symbols)
        if self.max_basis_functions and n_bfs > self.max_basis_functions:
            r = VQEResult(
                smiles, FAILED_ENERGY, 0, 0, VQEStatus.MOLECULE_TOO_LARGE,
                f'~{n_bfs} basis functions exceeds budget of {self.max_basis_functions}'
            )
            self._cache[smiles] = r
            return r
        try:
            H, n_qubits = _run_with_timeout(
                qml.qchem.molecular_hamiltonian,
                self.hamiltonian_timeout_sec,
                symbols, coords, charge=0, mult=1,
                basis=self.basis,
                active_electrons=self.active_electrons,
                active_orbitals=self.active_orbitals,
                load_data=False,
            )
        except TimeoutError as exc:
            r = VQEResult(smiles, FAILED_ENERGY, 0, 0, VQEStatus.TIMEOUT, str(exc))
            self._cache[smiles] = r
            return r
        except Exception as exc:
            r = VQEResult(smiles, FAILED_ENERGY, 0, 0, VQEStatus.HAMILTONIAN_FAILED, str(exc))
            self._cache[smiles] = r
            return r
        try:
            energy, steps = _run_with_timeout(
                self._optimize, self.optimization_timeout_sec, H, n_qubits
            )
        except TimeoutError as exc:
            r = VQEResult(smiles, FAILED_ENERGY, n_qubits, 0, VQEStatus.TIMEOUT, str(exc))
            self._cache[smiles] = r
            return r
        except Exception as exc:
            r = VQEResult(smiles, FAILED_ENERGY, n_qubits, 0, VQEStatus.OPTIMIZATION_FAILED, str(exc))
            self._cache[smiles] = r
            return r
        r = VQEResult(smiles, float(energy), n_qubits, steps, VQEStatus.OK)
        self._cache[smiles] = r
        return r

    def _optimize(self, H, n_qubits):
        dev = qml.device(self.device_name, wires=n_qubits)

        @qml.qnode(dev)
        def cost_fn(params):
            for i in range(n_qubits):
                qml.RY(params[i], wires=i)
            for layer in range(2):
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                for i in range(n_qubits):
                    qml.RY(params[n_qubits + layer * n_qubits + i], wires=i)
            return qml.expval(H)

        n_params = n_qubits * 3
        params = pnp.array(pnp.random.normal(0, pnp.pi, n_params), requires_grad=True)
        opt = qml.AdamOptimizer(stepsize=self.vqe_stepsize)
        last = float('inf')
        for _ in range(self.vqe_steps):
            params, last = opt.step_and_cost(cost_fn, params)
        return float(last), self.vqe_steps

print('VQE scorer ready.')

## 6. Choose your candidate set

Pick **one** of the two cells below — either sample new molecules from the
trained MolGAN, or paste a custom SMILES list.

In [ ]:
# --- Option A: sample from the trained generator ---
N_CANDIDATES = 30
candidates = molgan.sample_smiles(N_CANDIDATES)
print(f'Sampled {len(candidates)} unique valid SMILES:')
for s in candidates:
    print(' -', s)

In [ ]:
# --- Option B: use your own SMILES (overrides Option A if you run this cell) ---
# candidates = [
#     'CCO',
#     'c1ccccc1',
#     'CC(=O)O',
#     'CCN',
# ]
# print(f'Using {len(candidates)} custom SMILES.')

## 7. Score with VQE

Per-molecule wall-clock (Colab CPU, 6-qubit Hamiltonian, 50 VQE steps): roughly
**20–60 seconds**. Drop `VQE_STEPS` to 20 if you want a quick first pass.

In [ ]:
import time

ACTIVE_ELECTRONS = 2
ACTIVE_ORBITALS = 2
VQE_STEPS = 30
VQE_STEPSIZE = 0.1

# Defensive limits. Anything larger gets skipped instantly with a clear
# 'molecule_too_large' status instead of stalling SCF for many minutes.
MAX_HEAVY_ATOMS = 6
HAMILTONIAN_TIMEOUT_SEC = 60.0
OPTIMIZATION_TIMEOUT_SEC = 60.0

scorer = VQEScorer(
    active_electrons=ACTIVE_ELECTRONS,
    active_orbitals=ACTIVE_ORBITALS,
    vqe_steps=VQE_STEPS,
    vqe_stepsize=VQE_STEPSIZE,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
    hamiltonian_timeout_sec=HAMILTONIAN_TIMEOUT_SEC,
    optimization_timeout_sec=OPTIMIZATION_TIMEOUT_SEC,
)

# Up-front size filter so the user sees what's happening immediately.
scorable, skipped = [], []
for smi in candidates:
    ok, reason = scorer.screen(smi)
    (scorable if ok else skipped).append((smi, reason))

print(f'{len(scorable)} candidates within VQE budget; '
      f'{len(skipped)} skipped (too large for our (2e, 2o) demo).')
for smi, reason in skipped:
    print(f'  - SKIP {smi}  ({reason})')

results = []
started = time.time()
for i, (smi, _) in enumerate(scorable, start=1):
    print(f'[{i:>2}/{len(scorable)}] scoring {smi}...', flush=True)
    t0 = time.time()
    r = scorer.score(smi)
    dt = time.time() - t0
    tag = 'OK' if r.succeeded else r.status.value
    energy = f'{r.energy:+.4f}' if r.succeeded else '   --   '
    print(
        f'           --> {tag:<22s} energy={energy} qubits={r.n_qubits} ({dt:.1f}s)',
        flush=True,
    )
    results.append(r)

# Append skipped molecules to results so the CSV export still includes them.
for smi, reason in skipped:
    results.append(VQEResult(smi, FAILED_ENERGY, 0, 0, VQEStatus.MOLECULE_TOO_LARGE, reason))

total = time.time() - started
n = max(1, len(scorable))
print(f'\nDone in {total:.1f}s ({total / n:.1f}s per attempted molecule).')
print('Note: the first molecule is 2-4x slower due to one-time PySCF setup.')
print('Tip: raise MAX_HEAVY_ATOMS above to score larger molecules (longer runtime).')

## 8. Top candidates

In [ ]:
TOP_K = 5

successful = sorted((r for r in results if r.succeeded), key=lambda r: r.energy)
deduped, seen = [], set()
for r in successful:
    if r.smiles in seen:
        continue
    seen.add(r.smiles)
    deduped.append(r)
top = deduped[:TOP_K]

if not top:
    print('No successful VQE results.')
else:
    print(f'Top {len(top)} candidates by VQE ground-state energy:')
    for i, r in enumerate(top, start=1):
        print(f'  #{i}  {r.smiles:<35s} energy={r.energy:+.4f} Ha  qubits={r.n_qubits}')

## 9. Render the top molecules

In [ ]:
from rdkit.Chem import Draw

if top:
    mols = [Chem.MolFromSmiles(r.smiles) for r in top]
    legends = [f'#{i+1} | {r.energy:+.4f} Ha' for i, r in enumerate(top)]
    grid = Draw.MolsToGridImage(
        mols,
        molsPerRow=min(len(mols), 5),
        subImgSize=(260, 260),
        legends=legends,
    )
    display(grid)

## 10. Export results to CSV

In [ ]:
import pandas as pd

rows = [
    {
        'smiles': r.smiles,
        'energy_Ha': r.energy if r.succeeded else None,
        'qubits': r.n_qubits,
        'status': r.status.value,
        'error': r.error or '',
    }
    for r in results
]
df = pd.DataFrame(rows).sort_values('energy_Ha', na_position='last').reset_index(drop=True)
df.head(20)

In [ ]:
OUT_CSV = '/content/vqe_results.csv'
df.to_csv(OUT_CSV, index=False)
files.download(OUT_CSV)